# Jakarta Globe News Scraper - Jupyter Notebook

This notebook demonstrates web scraping of Jakarta Globe news articles and converts the results to pandas DataFrame format.

## Features:
- Scrapes articles from Jakarta Globe (Indonesian news portal)
- Configurable date ranges (defaults to last 2 days)
- Automatic pagination through article listings
- Prevents duplicate entries by tracking URLs
- Extracts full article content from individual pages
- Translates articles from Indonesian to English and/or Thai
- Exports to CSV/JSON formats

## Note:
If the website returns 403 Forbidden, try running from a different network or use Playwright (see cell below).

In [ ]:
# Install required packages
!pip install beautifulsoup4 lxml requests pandas cloudscraper deep-translator

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from datetime import datetime, timedelta
import time
import json
import os

import cloudscraper
from deep_translator import GoogleTranslator

In [ ]:
# Configuration
BASE_URL = "https://jakartaglobe.id/search/business/"
DEFAULT_DAYS_BACK = 2
MAX_PAGES = 100
REQUEST_DELAY = 0.5  # Delay between requests (seconds)

# Output files
OUTPUT_FILE = "news_data.json"
OUTPUT_EN_FILE = "news_data_translated_en.json"
OUTPUT_TH_FILE = "news_data_translated_th.json"

In [ ]:
class JakartaGlobeScraper:
    """Scraper class for Jakarta Globe news articles."""
    
    def __init__(self, days_back=DEFAULT_DAYS_BACK, fetch_content=False, translate_en=False, translate_th=False):
        self.days_back = days_back
        self.fetch_content = fetch_content
        self.translate_en = translate_en
        self.translate_th = translate_th
        self.existing_urls = set()
        self.articles = []
        self.cutoff_date = datetime.now() - timedelta(days=self.days_back)
        
        # Initialize translators
        self.translator_en = None
        self.translator_th = None
        if self.translate_en:
            self.translator_en = GoogleTranslator(source='id', target='en')
        if self.translate_th:
            self.translator_th = GoogleTranslator(source='id', target='th')
        
        print(f"Initialized scraper with {days_back} days back")
        print(f"Cutoff date: {self.cutoff_date}")
    
    def load_existing_data(self):
        """Load existing URLs from output file to prevent duplicates."""
        if os.path.exists(OUTPUT_FILE):
            try:
                with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
                    existing_data = json.load(f)
                    for article in existing_data:
                        if 'url' in article:
                            self.existing_urls.add(article['url'])
                print(f"Loaded {len(self.existing_urls)} existing URLs")
            except Exception as e:
                print(f"Could not load existing data: {e}")
    
    def get_page_url(self, page: int) -> str:
        """Generate URL for specific page."""
        if page == 1:
            return f"{BASE_URL}1"
        return f"{BASE_URL}{page}"

    def fetch_page(self, url: str) -> str:
        """Fetch HTML content from URL using cloudscraper."""
        try:
            scraper = cloudscraper.create_scraper(
                browser={'browser': 'chrome', 'platform': 'windows', 'mobile': False}
            )
            response = scraper.get(url, timeout=30)
            response.raise_for_status()
            return response.text
        except Exception as e:
            print(f"Error fetching {url}: {e}")
            return None
    def parse_article(self, elem, soup) -> dict:
        """Parse article from HTML element."""
        article = {}
        
        if elem.name == 'a':
            article['url'] = elem.get('href', '')
            title_elem = elem.select_one('h2, h3, h4, .title, .article-title')
            article['title'] = title_elem.get_text(strip=True) if title_elem else elem.get_text(strip=True)
        else:
            link_elem = elem.select_one('a[href*="/read/"]')
            if not link_elem:
                return None
            
            article['url'] = link_elem.get('href', '')
            title_elem = (elem.select_one('h2, h3, h4, .title, .article-title, .news-title') or 
                          link_elem.select_one('h2, h3, h4, .title'))
            article['title'] = title_elem.get_text(strip=True) if title_elem else link_elem.get_text(strip=True)
        
        if not article.get('url') or '/read/' not in article['url']:
            return None
        
        if not article['url'].startswith('http'):
            article['url'] = 'https://jakartaglobe.id' + article['url']
        
        date_match = re.search(r'/read/(\d{4})(\d{2})(\d{2})/', article['url'])
        if date_match:
            year, month, day = date_match.groups()
            article['date'] = f"{year}-{month}-{day}T00:00:00"
            article['date_parsed'] = datetime(int(year), int(month), int(day))
        else:
            article['date'] = datetime.now().strftime('%Y-%m-%dT%H:%M:%S')
            article['date_parsed'] = datetime.now()
        
        category_elem = elem.select_one('.category, .category-tag, span.category')
        article['category'] = category_elem.get_text(strip=True) if category_elem else 'Business'
        
        if article['url'] in self.existing_urls:
            article['is_existing'] = True
        else:
            article['is_existing'] = False
            self.existing_urls.add(article['url'])
        
        article['scraped_at'] = datetime.now().isoformat()
        
        return article

    def scrape_listing_page(self, page: int) -> list:
        """Scrape articles from a listing page."""
        url = self.get_page_url(page)
        print(f"Scraping page {page}: {url}")
        
        html = self.fetch_page(url)
        if not html:
            return []
        
        soup = BeautifulSoup(html, 'lxml')
        articles = []
        
        article_elements = (
            soup.select('div.article-item') or
            soup.select('div.news-item') or
            soup.select('div.article-card') or
            soup.select('article') or
            soup.select('div.latest-news article') or
            soup.select('div.search-result article') or
            soup.select('div.news-list article') or
            []
        )
        
        if not article_elements:
            article_elements = soup.select('a[href*="/read/"]')
        
        for elem in article_elements:
            article = self.parse_article(elem, soup)
            if article:
                articles.append(article)
        
        print(f"Found {len(articles)} articles on page {page}")
        return articles
    
    def should_stop_pagination(self, articles: list) -> bool:
        """Check if we should stop pagination based on article dates."""
        if not articles:
            return True
        
        for article in articles:
            if 'date_parsed' in article and isinstance(article['date_parsed'], datetime):
                if article['date_parsed'] >= self.cutoff_date:
                    return False
        return True
    
    def fetch_article_content(self, url: str) -> dict:
        """Fetch full article content."""
        content_data = {
            'content': '',
            'content_html': '',
            'author': '',
            'author_role': '',
            'published_time': '',
            'modified_time': '',
            'summary': '',
            'word_count': 0,
            'categories': [],
            'subcategory': '',
            'tags': [],
            'images': []
        }
        
        html = self.fetch_page(url)
        if not html:
            return content_data
        
        soup = BeautifulSoup(html, 'lxml')
        
        content_elem = (
            soup.select_one('div.article-content') or
            soup.select_one('div.content') or
            soup.select_one('div.article-body') or
            soup.select_one('div.post-content') or
            soup.select_one('article div.content') or
            soup.select_one('div.entry-content')
        )
        
        if content_elem:
            content_data['content_html'] = str(content_elem)
            paragraphs = content_elem.select('p')
            content_data['content'] = '\n\n'.join(
                p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)
            )
            if paragraphs and paragraphs[0].get_text(strip=True):
                content_data['summary'] = paragraphs[0].get_text(strip=True)
            content_data['word_count'] = len(content_data['content'].split())
        
        author_elem = (
            soup.select_one('meta[name="author"]') or
            soup.select_one('.author-name') or
            soup.select_one('.author') or
            soup.select_one('[rel="author"]')
        )
        if author_elem:
            content_data['author'] = author_elem.get('content', '') or author_elem.get_text(strip=True)
        
        role_elem = soup.select_one('.author-role, .author-title, .author-position')
        if role_elem:
            content_data['author_role'] = role_elem.get_text(strip=True)
        
        published_elem = soup.select_one('meta[property="article:published_time"]')
        if published_elem:
            content_data['published_time'] = published_elem.get('content', '')
        
        modified_elem = soup.select_one('meta[property="article:modified_time"]')
        if modified_elem:
            content_data['modified_time'] = modified_elem.get('content', '')
        
        breadcrumb = soup.select('nav.breadcrumb a, ol.breadcrumb a, .breadcrumb a')
        categories = []
        for crumb in breadcrumb:
            cat_text = crumb.get_text(strip=True)
            if cat_text and cat_text not in ['Home', 'Beranda', 'Jakarta Globe']:
                categories.append(cat_text)
        if categories:
            content_data['categories'] = categories
            content_data['subcategory'] = categories[-1] if categories else ''
        
        tags_elem = soup.select_one('meta[name="keywords"]')
        if tags_elem:
            tags_content = tags_elem.get('content', '')
            content_data['tags'] = [t.strip() for t in tags_content.split(',') if t.strip()]
        
        hero_img = soup.select_one('meta[property="og:image"]')
        if hero_img:
            hero_url = hero_img.get('content', '')
            if hero_url:
                content_data['images'].append({
                    'url': hero_url,
                    'alt': soup.select_one('meta[property="og:image:alt"]').get('content', '') if soup.select_one('meta[property="og:image:alt"]') else '',
                    'is_hero': True
                })
        
        if content_elem:
            for img in content_elem.select('img'):
                img_url = img.get('src', '') or img.get('data-src', '')
                if img_url and not img_url.startswith('data:'):
                    content_data['images'].append({
                        'url': img_url,
                        'alt': img.get('alt', ''),
                        'is_hero': False
                    })
        
        return content_data
    
    def translate_article(self, article: dict, target: str = 'en') -> dict:
        """Translate article to target language."""
        translator = self.translator_en if target == 'en' else self.translator_th
        if not translator:
            return article
        
        translated = article.copy()
        target_lang = 'en' if target == 'en' else 'th'
        
        translated['title_original'] = article.get('title', '')
        if article.get('content'):
            translated['content_original'] = article.get('content', '')
        if article.get('category'):
            translated['category_original'] = article.get('category', '')
        if article.get('author'):
            translated['author_original'] = article.get('author', '')
        
        try:
            if article.get('title'):
                translated['title'] = translator.translate(article['title'])
            
            if article.get('content'):
                content = article['content']
                if len(content) > 4000:
                    chunks = [content[i:i+4000] for i in range(0, len(content), 4000)]
                    translated_content = ' '.join(translator.translate(chunk) for chunk in chunks)
                else:
                    translated_content = translator.translate(content)
                translated['content'] = translated_content
            
            if article.get('category'):
                translated['category'] = translator.translate(article['category'])
            
            if article.get('author'):
                translated['author'] = translator.translate(article['author'])
            
            translated['translated_at'] = datetime.now().isoformat()
            translated['translation_source'] = 'id'
            translated['translation_target'] = target_lang
            
            print(f"Translated to {target_lang}: {translated.get('title', '')[:50]}...")
            
        except Exception as e:
            print(f"Translation error: {e}")
        
        return translated
    
    def save_articles(self):
        """Save articles to JSON file."""
        try:
            existing_data = []
            if os.path.exists(OUTPUT_FILE):
                with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
                    existing_data = json.load(f)
            
            existing_urls = {a.get('url') for a in existing_data}
            for article in self.articles:
                if article.get('url') not in existing_urls:
                    existing_data.append(article)
            
            with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
                json.dump(existing_data, f, indent=2, ensure_ascii=False)
            
            print(f"Saved {len(self.articles)} new articles to {OUTPUT_FILE}")
            
        except Exception as e:
            print(f"Error saving articles: {e}")
    
    def save_translated_articles(self, articles: list, filename: str):
        """Save translated articles to JSON file."""
        try:
            with open(filename, 'w', encoding='utf-8') as f:
                json.dump(articles, f, indent=2, ensure_ascii=False)
            print(f"Saved {len(articles)} translated articles to {filename}")
        except Exception as e:
            print(f"Error saving translated articles: {e}")
    
    def scrape(self):
        """Main scraping method."""
        print("Starting Jakarta Globe scraper...")
        
        self.load_existing_data()
        
        page = 1
        total_new = 0
        
        while page <= MAX_PAGES:
            articles = self.scrape_listing_page(page)
            
            if not articles:
                print("No articles found, stopping pagination")
                break
            
            if self.should_stop_pagination(articles):
                print("All articles are older than cutoff date, stopping pagination")
                for article in articles:
                    if article.get('date_parsed') and article['date_parsed'] >= self.cutoff_date:
                        if not article.get('is_existing'):
                            self.articles.append(article)
                            total_new += 1
                break
            
            for article in articles:
                if not article.get('is_existing'):
                    self.articles.append(article)
                    total_new += 1
            
            print(f"Page {page}: {len(articles)} articles, {sum(1 for a in articles if not a.get('is_existing'))} new")
            
            page += 1
            time.sleep(REQUEST_DELAY)
        
        print(f"Found {total_new} new articles from {page - 1} pages")
        
        if self.fetch_content and self.articles:
            print("Fetching article content...")
            for i, article in enumerate(self.articles):
                print(f"Fetching content {i+1}/{len(self.articles)}: {article.get('title', '')[:50]}...")
                content_data = self.fetch_article_content(article['url'])
                article.update(content_data)
                time.sleep(REQUEST_DELAY)
        
        if self.articles:
            self.save_articles()
        
        if self.translate_en and self.articles:
            print("Translating articles to English...")
            translated_en = []
            for article in self.articles:
                translated = self.translate_article(article, 'en')
                translated_en.append(translated)
                time.sleep(0.5)
            self.save_translated_articles(translated_en, OUTPUT_EN_FILE)
        
        if self.translate_th and self.articles:
            print("Translating articles to Thai...")
            translated_th = []
            for article in self.articles:
                translated = self.translate_article(article, 'th')
                translated_th.append(translated)
                time.sleep(0.5)
            self.save_translated_articles(translated_th, OUTPUT_TH_FILE)
        
        print("Scraping completed!")
        return self.articles

In [ ]:
# Initialize scraper - CONFIGURE OPTIONS HERE
scraper = JakartaGlobeScraper(
    days_back=2,           # Number of days back to scrape
    fetch_content=False,   # Fetch full article content (slower)
    translate_en=False,    # Translate to English
    translate_th=False    # Translate to Thai
)

In [ ]:
# Run the scraper
all_articles = scraper.scrape()

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(all_articles)

print("DataFrame shape:", df.shape)
print("\nColumn names:", df.columns.tolist())

In [ ]:
# Display the DataFrame
df

In [ ]:
# Display first few rows
df.head(10)

In [ ]:
# Display article titles
print("Article Titles:\n")
for i, title in enumerate(df['title'].head(10), 1):
    print(f"{i}. {title}")

In [ ]:
# Show date distribution
print("Date distribution:")
print(df['date'].value_counts())

In [ ]:
# Show category distribution
print("Category distribution:")
print(df['category'].value_counts())

In [ ]:
# Save to CSV
df.to_csv('jktglobe_news.csv', index=False)
print("Data saved to jktglobe_news.csv")

In [ ]:
# Display summary for each article (if content was fetched)
if 'content' in df.columns:
    for i, row in df.iterrows():
        print(f"\n{'='*60}")
        print(f"Title: {row['title']}")
        print(f"URL: {row['url']}")
        print(f"Date: {row['date']}")
        print(f"Author: {row.get('author', 'N/A')}")
        print(f"Summary: {row.get('summary', 'N/A')[:200]}...")
        print(f"Content length: {len(row.get('content', ''))} chars")
        if i >= 4:  # Show first 5
            break

## Alternative: Using Playwright (for 403 errors)

If the website returns 403 Forbidden with cloudscraper, try using Playwright:

```python
# Install playwright
!pip install playwright
!playwright install chromium

from playwright.sync_api import sync_playwright

def fetch_page_playwright(url: str) -> str:
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(url, wait_until='networkidle', timeout=30000)
        html = page.content()
        browser.close()
        return html
```